# Lesson 03: What's Changed in MCP 2026-07-28

In this notebook, you will:
1. Construct a self-contained `2026-07-28` stateless MCP request with `_meta` capabilities and `traceparent`.
2. Enforce `Mcp-Method` / `Mcp-Name` header-to-body integrity.
3. Run a multi-round-trip `InputRequiredResult` elicitation workflow and validate Full JSON Schema 2020-12 `$defs`.

In [ ]:
!pip install -q -U jsonschema

In [ ]:
import base64
import json
import uuid
from jsonschema import validate

# 1. Stateless 2026-07-28 request envelope
request_envelope = {
    "headers": {
        "MCP-Protocol-Version": "2026-07-28",
        "Mcp-Method": "tools/call",
        "Mcp-Name": "add_item",
    },
    "body": {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": {
            "name": "add_item",
            "arguments": {"basket_id": "b_" + uuid.uuid4().hex[:6], "item": "MCP Handbook"},
            "_meta": {
                "io.modelcontextprotocol/protocolVersion": "2026-07-28",
                "io.modelcontextprotocol/clientInfo": {"name": "colab-client", "version": "1.0"},
                "traceparent": "00-4bf92f3577b34da6a3ce929d0e0e4736-00f067aa0ba902b7-01",
            },
        },
    },
}
print(json.dumps(request_envelope, indent=2))

In [ ]:
# 2. Full JSON Schema 2020-12 tool input validation with $defs
tool_input_schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "$defs": {
        "BasketId": {"type": "string", "pattern": "^b_[a-f0-9]{6}$"}
    },
    "properties": {
        "basket_id": {"$ref": "#/$defs/BasketId"},
        "item": {"type": "string", "minLength": 1},
    },
    "required": ["basket_id", "item"],
}

validate(instance=request_envelope["body"]["params"]["arguments"], schema=tool_input_schema)
print("JSON Schema 2020-12 validation passed!")

### Challenge Exercise
Write a cache manager that stores `tools/list` responses with `ttlMs` and `cacheScope`, returning cached entries only when `(now_ms - stored_ms) < ttlMs` and `cacheScope == 'public'` (or matching `caller_id` when `'private'`).